# Predictive analysis of naval incidents in the USA, 2002 - 2015: <br>
## Model deployment: Server examples for VesselBalancedSample

> Author: [Oscar Anton](https://www.linkedin.com/in/oscanton/) <br>
> Date: 2024 <br>
> License: [CC BY-NC-ND 4.0 DEED](https://creativecommons.org/licenses/by-nc-nd/4.0/) <br>
> Version: 0.9 <br>

# 0. Loadings

### Libraries

In [1]:
import pandas as pd
import joblib

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn

from typing import List
from fastapi.middleware.cors import CORSMiddleware

import threading

### Models

In [2]:
# Models folder
models_path = '../5.DataModel/Models/'

# Load Random Forest model
rf_MA_model = joblib.load(models_path + 'rf_MA_train.pkl')

# Check parameters
params = rf_MA_model.get_params()
for param_name, param_value in params.items():
    print(f" {param_name}: {param_value}")

 bootstrap: True
 ccp_alpha: 0.0
 class_weight: None
 criterion: gini
 max_depth: None
 max_features: sqrt
 max_leaf_nodes: None
 max_samples: None
 min_impurity_decrease: 0.0
 min_samples_leaf: 1
 min_samples_split: 2
 min_weight_fraction_leaf: 0.0
 monotonic_cst: None
 n_estimators: 100
 n_jobs: None
 oob_score: False
 random_state: 42
 verbose: 0
 warm_start: False


In [3]:
# Load KNeighborsRegressor model
actid_knn_model = joblib.load(models_path + 'actid_knn_model.pkl')

# Check parameters
params = actid_knn_model.get_params()
for param_name, param_value in params.items():
    print(f" {param_name}: {param_value}")

 algorithm: auto
 leaf_size: 30
 metric: minkowski
 metric_params: None
 n_jobs: None
 n_neighbors: 10
 p: 2
 weights: uniform


### Scaler

In [4]:
# Load used scaler for the trainings
datasets_path = '../5.DataModel/Datasets/'
scaler = joblib.load(datasets_path + 'scaler_MA.pkl')

# Check scalable variables
print(f'Scalable variables: {scaler.feature_names_in_}')

Scalable variables: ['activity_id' 'hour' 'longitude' 'age' 'gross_ton' 'vessel_length'
 'air_temp' 'wind_speed']


# 1. Variable transformation & encoding

### Input data validation

In [5]:
# Define expected columns entered by the user
expected_columns = {
    'region': str,
    'watertype': str,
    'damage_status': str,
    'vessel_class': str,
    #'activity_id': int,
    'hour': str,
    'longitude': float,
    'age': (int, float),
    'gross_ton': (int, float),
    'vessel_length': (int, float),
    'air_temp': (int, float),
    'wind_speed': (int, float)
}

# Columns and data types comparations
def input_evaluation(df):
    # Check expect columns and type
    for column, column_type in expected_columns.items():
        if column not in df.columns:
            return f"Missing expected columns: {column}"
        if not df[column].apply(lambda x: isinstance(x, column_type)).all():
            return f"Column {column} should be of type {column_type}"
    
    # Check hour column format
    try:
        pd.to_datetime(df['hour'], format='%H:%M')
    except ValueError:
        return "'hour' should be 'HH:MM' format"

    return True

### Input data formating

In [6]:
# Define expected variables by the model
input_columns = [
    'region_Alaska', 'region_Canada', 'region_East Coast',
    'region_Gulf of Mexico', 'region_Mississippi', 'region_West Coast',
    'watertype_ocean', 'watertype_river', 'damage_status_Actual Total Loss',
    'damage_status_Damaged',
    'damage_status_Total Constructive Loss: Salvaged',
    'damage_status_Total Constructive Loss: Unsalvaged',
    'damage_status_Undamaged', 'vessel_class_Barge',
    'vessel_class_Bulk Carrier', 'vessel_class_Fishing Vessel',
    'vessel_class_General Dry Cargo Ship',
    'vessel_class_Miscellaneous Vessel', 'vessel_class_Offshore',
    'vessel_class_Passenger Ship', 'vessel_class_Recreational',
    'vessel_class_Tank Ship', 'vessel_class_Towing Vessel',
    'vessel_class_other value',
    #'activity_id',
    'hour', 'longitude', 'age',
    'gross_ton', 'vessel_length', 'air_temp', 'wind_speed'
]

# Transform categorical variables to one hot codifing
def value_ohe(variable_name, values):
    one_hot_df = []
    labels = list(filter(lambda x: x.startswith(variable_name), input_columns))
    for value in values:
        target_column = variable_name + '_' + value
        one_hot_row = pd.DataFrame(0, index=[0], columns=labels)
        one_hot_row[target_column] = 1
        one_hot_df.append(one_hot_row)
    # Concatenate all one-hot encoded rows into a single DataFrame
    return pd.concat(one_hot_df, ignore_index=True)

# Transform minutes to decimal value
def hour_dec(hour):
    return (pd.to_numeric(hour.str.split(':').str[0]) + 
            pd.to_numeric(hour.str.split(':').str[1])/60).round(2)

# Transform numeric variables to scaled values
def scale_values(scaler, feature_names, **kwargs):
    # Create a DataFrame with the new entries
    new_data = pd.DataFrame(kwargs)

    # Transform the data with the scaler adjusted
    scaled_values = pd.DataFrame(scaler.transform(new_data), columns=feature_names)
    
    return scaled_values

# Define structure of data input
def input_structure(df):
    data_input = pd.concat([
        value_ohe('region', df['region']),
        value_ohe('watertype', df['watertype']),
        value_ohe('damage_status', df['damage_status']),
        value_ohe('vessel_class', df['vessel_class']),
        scale_values(scaler,
                     feature_names = ['activity_id', 'hour', 'longitude', 'age',
                                      'gross_ton', 'vessel_length', 'air_temp', 'wind_speed'],
                     activity_id = [0] * len(df),
                     hour = hour_dec(df['hour']),
                     longitude = df['longitude'],
                     age = df['age'],
                     gross_ton = df['gross_ton'],
                     vessel_length = df['vessel_length'],
                     air_temp = df['air_temp'],
                     wind_speed = df['wind_speed'],
                     )
        ], axis=1)
    return data_input

# 2. Predictions for new data examples

In [7]:
# Load example dataframe
X_test = pd.read_hdf(datasets_path + 'datasets_MA_splited.h5', key = 'X_test')

# Check first observation
X_test.iloc[[0]]

,region_Alaska,region_Canada,region_East Coast,region_Gulf of Mexico,region_Mississippi,region_West Coast,watertype_ocean,watertype_river,damage_status_Actual Total Loss,damage_status_Damaged,...,vessel_class_Towing Vessel,vessel_class_other value,activity_id,hour,longitude,age,gross_ton,vessel_length,air_temp,wind_speed
37979,1,0,0,0,0,0,1,0,0,1,...,0,0,-1.004202,1.739413,-2.435766,-0.612438,3.069414,3.115062,-0.971461,1.804097


In [8]:
# Inverse transform: from scaled values to original ones
inv_activity_id = scaler.inverse_transform([[-1.004202, 1.739413, -2.435766, -0.612438, 3.069414, 3.115062, -0.971461, 1.804097]])
pd.DataFrame(inv_activity_id.round(3), columns=scaler.feature_names_in_)

,activity_id,hour,longitude,age,gross_ton,vessel_length,air_temp,wind_speed
0,2250545.158,22.5,-154.75,16.0,50205.007,863.3,65.531,102.781


In [9]:
# Same for 10th observation
X_test.iloc[[10]]

,region_Alaska,region_Canada,region_East Coast,region_Gulf of Mexico,region_Mississippi,region_West Coast,watertype_ocean,watertype_river,damage_status_Actual Total Loss,damage_status_Damaged,...,vessel_class_Towing Vessel,vessel_class_other value,activity_id,hour,longitude,age,gross_ton,vessel_length,air_temp,wind_speed
43684,0,0,0,1,0,0,1,0,0,1,...,0,0,1.132198,-0.413976,0.036705,1.415134,-0.345464,-0.755818,0.911033,-0.089868


In [10]:
# Inverse transform: from scaled values to original ones
inv_activity_id = scaler.inverse_transform([[1.132198, -0.413976, 0.036705, 1.415134, -0.345464, -0.755818, 0.911033, -0.089868]])
pd.DataFrame(inv_activity_id.round(3), columns=scaler.feature_names_in_)

,activity_id,hour,longitude,age,gross_ton,vessel_length,air_temp,wind_speed
0,4318926.568,9.25,-94.884,50.0,13.006,32.7,233.458,48.167


In [11]:
# Example values (for checking purposes, example = X_test.iloc[[0]])
input = pd.DataFrame({
    "region": ["Alaska"],
    "watertype": ["ocean"],
    "damage_status": ["Damaged"],
    "vessel_class": ["General Dry Cargo Ship"],
    #"activity_id": [2250545],
    "hour": ["22:30"],
    "longitude": [-154.75],
    "age": [16],
    "gross_ton": [20205],
    "vessel_length": [863.3],
    "air_temp": [65.531],
    "wind_speed": [102.781]
})

In [12]:
# Example values (for checking purposes, example = X_test.iloc[[0]] & X_test.iloc[[10]])
input_df = pd.DataFrame({
    "region": ["Alaska", "Gulf of Mexico"],
    "watertype": ["ocean", "ocean"],
    "damage_status": ["Damaged", "Damaged"],
    "vessel_class": ["General Dry Cargo Ship", "Fishing Vessel"],
    #"activity_id": [2250545, 4318926],
    "hour": ["22:30", "9:15"],
    "longitude": [-154.75, -94.884],
    "age": [16, 50],
    "gross_ton": [20205, 13],
    "vessel_length": [863.3, 32.7],
    "air_temp": [65.531, 233.458],
    "wind_speed": [102.781, 48.167]
})

In [13]:
def get_prediction_probabilities(input_df):
    if input_evaluation(input_df) == True:
        try:
            # Format input data
            structured_input = input_structure(input_df)

            # Replace activity_id for its predictions, according to knn model
            structured_input['activity_id'] = actid_knn_model.predict(structured_input.drop(columns='activity_id'))

            # Calculate incident probabilities array (output)
            prediction_proba = rf_MA_model.predict_proba(structured_input)
            
            # Store probabilities for each vessel in a dictionary
            event_classes = ['Critical Events', 'Maritime Accidents', 'Material Issues', 'Onboard Emergencies', 'Thirdparty Damages']
            result = {
                key: [f"{prediction_proba[j][i]:.4%}" for j in range(len(prediction_proba))]
                for i, key in enumerate(event_classes)
            }
            return result
        except ValueError as e:
            return f"Calculation of prediction probabilities failed: {e}"

    else:
        return f"Data evaluation failed: {input_evaluation(input_df)}"

In [14]:
get_prediction_probabilities(input)

{'Critical Events': ['22.0833%'],
 'Maritime Accidents': ['6.0000%'],
 'Material Issues': ['57.9167%'],
 'Onboard Emergencies': ['7.0000%'],
 'Thirdparty Damages': ['7.0000%']}

In [15]:
get_prediction_probabilities(input_df)

{'Critical Events': ['22.0833%', '28.0000%'],
 'Maritime Accidents': ['6.0000%', '27.0000%'],
 'Material Issues': ['57.9167%', '15.5000%'],
 'Onboard Emergencies': ['7.0000%', '9.5000%'],
 'Thirdparty Damages': ['7.0000%', '20.0000%']}

# 3. FastAPI POST requests

### Event class prediction

Uvicorn server running in 127.0.0.2:8000/predict

In [16]:
# Initiate FastAPI
app = FastAPI()

# Configure CORS to allow requests from any source
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # Allows all origins. Change this in production.
    allow_credentials=True,
    allow_methods=["*"],  # Allows all HTTP methods
    allow_headers=["*"],  # Allow all headers
)

# Define the structure of the request data using Pydantic
class PredictionRequest(BaseModel):
    region: List[str]
    watertype: List[str]
    damage_status: List[str]
    vessel_class: List[str]
    #activity_id: List[int]
    hour: List[str]
    longitude: List[float]
    age: List[float]
    gross_ton: List[float]
    vessel_length: List[float]
    air_temp: List[float]
    wind_speed: List[float]


@app.post('/predict')
def deploy_model(request: PredictionRequest):
    try:
        # Convert the request data to a DataFrame
        input_df = pd.DataFrame({
            'region': request.region,
            'watertype': request.watertype,
            'damage_status': request.damage_status,
            'vessel_class': request.vessel_class,
            #'activity_id': request.activity_id,
            'hour': request.hour,
            'longitude': request.longitude,
            'age': request.age,
            'gross_ton': request.gross_ton,
            'vessel_length': request.vessel_length,
            'air_temp': request.air_temp,
            'wind_speed': request.wind_speed
        })
        
        # Call the prediction function
        probabilities = get_prediction_probabilities(input_df)
        
        # Return the prediction probabilities
        return probabilities
    
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# Define a function to run the Uvicorn server
def run_uvicorn():
    uvicorn.run(app, host="127.0.0.2", port=8000, log_level="info")

# Start the server in a separate thread
if __name__ == "__main__":
    thread = threading.Thread(target=run_uvicorn)
    thread.start()

Checking available at: <br>
https://web.postman.co/workspace/My-Workspace~dcdd54f9-b9a9-48b6-8696-4024e746d3b6/request/37058628-1f643958-35e2-45f7-b8da-8a9122e76860?action=share&source=copy-link&creator=37058628